In [2]:
import pandas as pd

In [4]:
emp = pd.read_csv("../data/raw/eurostat/lfsa_egai2d.tsv", sep="\t")

key = emp.columns[0]
codes = emp[key].str.split(",", expand=True)
codes.columns = ["freq", "isco08", "age", "sex", "unit", "geo"]
emp = pd.concat([codes, emp.drop(columns=key)], axis=1)
emp = emp.melt(id_vars=["freq", "isco08", "age", "sex", "unit", "geo"],
               var_name="year", value_name="raw")
emp["year"] = emp["year"].str.strip().astype(int)


emp["value"] = pd.to_numeric(emp["raw"].str.extract(r"([\d.]+)")[0], errors="coerce")
emp["flag"] = emp["raw"].str.replace(r"[\d.:\s]", "", regex=True)
emp = emp[
    (emp["sex"] == "T") &                                  # both sexes
    (emp["age"] == "Y_GE15") &                             # everyone aged 15+
    (emp["isco08"].str.len() == 4) &                       # two-digit groups like OC25
    (~emp["isco08"].isin(["OC01", "OC02", "OC03"])) &      # drop armed forces
    (~emp["geo"].isin(["EA21", "EU27_2020"])) &            # drop EU-wide totals
    (emp["year"].isin([2015, 2024, 2025]))
]


emp["share"] = emp["value"] / emp.groupby(["geo", "year"])["value"].transform("sum")

In [5]:
print(emp.groupby(["geo", "year"])["share"].sum().round(6).value_counts())
print(emp[emp["year"] == 2025].groupby("geo")["value"].count().sort_values().head(10))

share
1.0    103
0.0      5
Name: count, dtype: int64
geo
ME     0
UK     0
LU    30
CY    36
IS    36
EE    36
BE    37
IE    37
MT    37
NO    37
Name: value, dtype: int64


In [6]:
emp[(emp["geo"] == "IT") & (emp["year"] == 2025)].sort_values("share", ascending=False)[["isco08", "share"]].head(5)

,isco08,share
353400,OC33,0.074442
357960,OC52,0.074404
357504,OC51,0.059455
355224,OC41,0.059203
350208,OC23,0.056359


In [7]:
emp.to_csv("../data/processed/employment_shares.csv", index=False)